In [1]:
import numpy as np

class MonteCarlo_SEIRS:
    def __init__(self, populations, W, states_init=None):
        """
        populations: Array of population sizes for each node [cite: 369]
        W: Mobility weight matrix (W_ij is flow from i to j) [cite: 34]
        """
        self.num_nodes = len(populations)
        self.total_agents = np.sum(populations)
        self.W = W

        # Create agents: each has a residence node and a current epidemic state
        # State mapping: 0=S, 1=E, 2=I, 3=R
        self.residence = np.repeat(np.arange(self.num_nodes), populations)
        if states_init is not None:
            self.states = states_init
        else:
            self.states = np.zeros(self.total_agents, dtype=int)
            # Initial seed: 1% infected [cite: 371]
            seed_idx = np.random.choice(self.total_agents, int(0.01 * self.total_agents), replace=False)
            self.states[seed_idx] = 2

    def get_destinations(self, pd):
        """Diffusion phase: Agents decide where to go for the day [cite: 373, 377]"""
        current_locs = self.residence.copy()

        # Identify who moves [cite: 374]
        movers_mask = np.random.rand(self.total_agents) < pd

        for i in range(self.num_nodes):
            node_movers = (self.residence == i) & movers_mask
            num_movers = np.sum(node_movers)

            if num_movers > 0:
                # Probability of moving to node j from i [cite: 376]
                probs = self.W[i] / np.sum(self.W[i])
                destinations = np.random.choice(self.num_nodes, size=num_movers, p=probs)
                current_locs[node_movers] = destinations

        return current_locs

    def run_step(self, pd, beta, sigma, gamma, xi):
        """One full day of the simulation """
        # 1. Commuting [cite: 373]
        current_locs = self.get_destinations(pd)

        # 2. Reaction (Contagion) [cite: 378, 379]
        new_states = self.states.copy()

        for node in range(self.num_nodes):
            # Find agents currently in this patch
            agents_in_patch = (current_locs == node)
            patch_states = self.states[agents_in_patch]

            num_I = np.sum(patch_states == 2) # Count Infected in patch
            if num_I > 0:
                # Probability of a Susceptible in this patch becoming Exposed
                # P(infection) = 1 - (1 - beta)^num_I
                p_inf = 1 - (1 - beta)**num_I

                susceptible_in_patch = agents_in_patch & (self.states == 0)
                infect_mask = np.random.rand(np.sum(susceptible_in_patch)) < p_inf

                # S -> E
                indices = np.where(susceptible_in_patch)[0]
                new_states[indices[infect_mask]] = 1

        # 3. Disease Progression (Agent-level transitions)
        # E -> I (incubation)
        exposed_mask = (self.states == 1)
        become_I = np.random.rand(np.sum(exposed_mask)) < sigma
        new_states[np.where(exposed_mask)[0][become_I]] = 2

        # I -> R (recovery) [cite: 380]
        infected_mask = (self.states == 2)
        become_R = np.random.rand(np.sum(infected_mask)) < gamma
        new_states[np.where(infected_mask)[0][become_R]] = 3

        # R -> S (loss of immunity)
        recovered_mask = (self.states == 3)
        become_S = np.random.rand(np.sum(recovered_mask)) < xi
        new_states[np.where(recovered_mask)[0][become_S]] = 0

        self.states = new_states
        # Return home happens implicitly because current_locs isn't saved

    def get_total_incidence(self):
        """Returns the fraction of agents that are currently Infected """
        return np.sum(self.states == 2) / self.total_agents

In [2]:
import torch
import numpy as np

def run_simulation_on_pyg(data):
    # Extract populations (first column of x: [num_nodes, 3])
    populations = data.x[:, 0].cpu().numpy().astype(int)
    num_nodes = len(populations)

    # Extract Mobility Matrix (W)
    # Create an empty dense matrix and fill it using edge_index and edge_attr
    W = np.zeros((num_nodes, num_nodes))
    rows = data.edge_index[0].cpu().numpy()
    cols = data.edge_index[1].cpu().numpy()
    weights = data.edge_attr.squeeze().cpu().numpy()

    for r, c, w in zip(rows, cols, weights):
        W[r, c] = w

    return populations, W

In [6]:
def find_threshold_mc(data, pd, sigma, gamma, xi):
    populations, W = run_simulation_on_pyg(data)

    # We sweep beta to find the threshold
    beta_range = np.linspace(0.001, 0.5, 20)

    for beta in beta_range:
        # Initialize the Monte Carlo simulator we built earlier
        sim = MonteCarlo_SEIRS(populations, W)

        # Run for a fixed number of steps to check for endemicity
        # Or use the paper's convergence rule
        stable_steps = 0
        prev_incidence = 0

        for t in range(500): # Max steps
            sim.run_step(pd, beta, sigma, gamma, xi)
            current_incidence = sim.get_total_incidence()

            # Check for convergence [cite: 383]
            if abs(current_incidence - prev_incidence) < 1e-5:
                stable_steps += 1
                print(stable_steps)
            else:
                stable_steps = 0

            prev_incidence = current_incidence

            if stable_steps >= 100:
                break

        # If the steady-state incidence is > 0, we found the threshold!
        if prev_incidence > 0.001:
            return beta

    return 1.0 # Default if never reached

In [4]:
real_graph = torch.load('/Users/emmapinckers/PycharmProjects/thesis/data generation/datasets/version1/real_graph.pt', weights_only=False)

In [7]:
# --- Example Integration ---
# Parameters for SEIRS
pd = 1     # Mobility probability [cite: 34]
sigma = 0.2  # 1/incubation (5 days)
gamma = 1  # recovery rate (10 days)
xi = 0.01    # Loss of immunity

# 1. Run the MC Search
simulated_lambda_c = find_threshold_mc(real_graph, pd, sigma, gamma, xi)



1


KeyboardInterrupt: 